[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/05_neuroevolution_genetic_algorithm.ipynb)

# Bài 05 — Thuật toán di truyền và neuroevolution

**🎯 Mục tiêu**
- **Thuật toán tiến hoá**: tối ưu bằng quần thể, chọn lọc, lai ghép, đột biến — **không cần gradient**.
- Cài **thuật toán di truyền** (GA) from scratch cho bài toán cái túi; so với lời giải chính xác (quy hoạch động), thuật toán tham lam và tìm kiếm ngẫu nhiên cùng ngân sách.
- **Neuroevolution**: tiến hoá trực tiếp trọng số chính sách cho CartPole (GA, tìm kiếm ngẫu nhiên, **Evolution Strategies**) — so với policy gradient về số bước tương tác.
- Khi nào nên (và không nên) dùng tiến hoá.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.envs import CartPole
from ml_course.visualization import set_style

set_style()

## 1. Ý tưởng

Thuật toán tiến hoá mô phỏng chọn lọc tự nhiên trên một **quần thể** lời giải:
1. **Đánh giá** độ thích nghi (fitness) của mỗi cá thể — chỉ cần *tính được* hàm mục tiêu, không cần đạo hàm.
2. **Chọn lọc** cá thể tốt làm bố mẹ (vd *tournament*: bốc ngẫu nhiên vài cá thể, lấy cá thể tốt nhất).
3. **Lai ghép** (crossover): trộn gen của hai bố mẹ.
4. **Đột biến** (mutation): thay đổi ngẫu nhiên nhỏ — duy trì đa dạng, khám phá.
5. **Tinh hoa** (elitism): giữ nguyên vài cá thể tốt nhất sang thế hệ sau.

Phù hợp với hàm mục tiêu **không khả vi, rời rạc, nhiều cực trị, hoặc là hộp đen** (mô phỏng, phần cứng, cấu hình).

## 2. Thuật toán di truyền cho bài toán cái túi

50 món đồ, mỗi món có khối lượng $w_i$ và giá trị $v_i$; chọn tập con có tổng khối lượng ≤ sức chứa, tổng giá trị lớn nhất. Đây là bài NP-khó nói chung; với khối lượng nguyên nhỏ, quy hoạch động cho **lời giải chính xác** để đối chiếu. Bộ dữ liệu "tương quan mạnh" ($v_i = w_i + 25$) — kiểu khó cho thuật toán tham lam.

Biểu diễn: chuỗi bit dài 50 (món $i$ có được chọn không). Fitness = tổng giá trị, trừ phạt nếu quá tải.

In [ ]:
rng = np.random.default_rng(2)
N_ITEMS = 50
weights = rng.integers(10, 100, N_ITEMS)
values = weights + 25
capacity = int(weights.sum() * 0.3)

dp = np.zeros(capacity + 1, dtype=int)                                 # quy hoạch động: lời giải chính xác
for i in range(N_ITEMS):
    for c in range(capacity, weights[i] - 1, -1):
        dp[c] = max(dp[c], dp[c - weights[i]] + values[i])
optimum = dp[capacity]

greedy_value, load = 0, 0
for i in np.argsort(-values / weights):                                # tham lam theo tỉ lệ giá trị/khối lượng
    if load + weights[i] <= capacity:
        load += weights[i]
        greedy_value += values[i]


def fitness(population):
    load, value = population @ weights, population @ values
    return np.where(load <= capacity, value, value - 10 * (load - capacity))


print(f"sức chứa {capacity} | tối ưu (DP) = {optimum} | tham lam = {greedy_value} ({greedy_value / optimum:.1%} tối ưu)")

In [ ]:
def genetic_algorithm(seed, pop_size=100, generations=200, mutation_rate=0.02, n_elite=4, tournament=3):
    r = np.random.default_rng(seed)
    population = (r.random((pop_size, N_ITEMS)) < 0.3).astype(int)
    best_per_generation = []
    for _ in range(generations):
        f = fitness(population)
        best_per_generation.append(f.max())
        children = [population[i] for i in np.argsort(-f)[:n_elite]]           # tinh hoa
        while len(children) < pop_size:
            def select():                                                      # tournament selection
                idx = r.integers(pop_size, size=tournament)
                return population[idx[np.argmax(f[idx])]]
            mom, dad = select(), select()
            child = np.where(r.random(N_ITEMS) < 0.5, mom, dad)                  # lai ghép đều (uniform)
            child = np.where(r.random(N_ITEMS) < mutation_rate, 1 - child, child)   # đột biến lật bit
            children.append(child)
        population = np.array(children)
    return fitness(population).max(), np.array(best_per_generation)


budget = 100 * 200                                                     # số lần đánh giá fitness của GA
random_best = fitness((np.random.default_rng(9).random((budget, N_ITEMS)) < 0.3).astype(int)).max()
print(f"tìm kiếm ngẫu nhiên ({budget:,} lời giải): {random_best} ({random_best / optimum:.1%})")
fig, ax = plt.subplots(figsize=(7.5, 4))
for seed in range(3):
    best, history = genetic_algorithm(seed)
    ax.plot(history / optimum, label=f"GA seed {seed}")
    print(f"GA seed {seed}: {best} ({best / optimum:.1%} tối ưu)")
ax.axhline(greedy_value / optimum, color="gray", ls="--", label="tham lam")
ax.axhline(random_best / optimum, color="gray", ls=":", label="ngẫu nhiên (cùng ngân sách)")
ax.set_ylim(0.85, 1.005)
ax.set_xlabel("thế hệ")
ax.set_ylabel("fitness tốt nhất / tối ưu")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

GA tìm được (hoặc gần như) lời giải tối ưu, vượt thuật toán tham lam và vượt xa tìm kiếm ngẫu nhiên với **cùng số lần đánh giá** — chọn lọc + lai ghép tập trung tìm kiếm vào vùng tốt, đột biến giữ đa dạng. Nhưng GA không **chứng minh** được tối ưu, và khi có cấu trúc đặc biệt (như DP ở đây) thì thuật toán chuyên biệt nhanh và chính xác hơn — GA hữu ích khi không có lựa chọn đó.

### Ảnh hưởng của tỉ lệ đột biến

In [ ]:
for rate in [0.0, 0.002, 0.02, 0.2]:
    results = [genetic_algorithm(seed, mutation_rate=rate)[0] for seed in range(3)]
    print(f"tỉ lệ đột biến {rate:5.3f}: kết quả 3 seed = {results} → trung bình {np.mean(results) / optimum:.1%} tối ưu")

Không đột biến hoặc đột biến rất nhỏ: quần thể nhanh chóng **hội tụ sớm** (mọi cá thể gần giống nhau, lai ghép không tạo gì mới) và kẹt ở lời giải tốt-nhưng-chưa-tối-ưu — khác nhau tuỳ seed. Đột biến vừa phải (0.02) tìm ra tối ưu ở mọi seed. Đột biến quá lớn (0.2): con cái gần như ngẫu nhiên — mất thông tin di truyền từ bố mẹ, kết quả kém hẳn. Đây chính là cân bằng **khám phá–khai thác** (bài 01) dưới một dạng khác.

## 3. Neuroevolution: tiến hoá chính sách cho CartPole

Thay vì tính gradient (bài 04), coi **return của một lượt chơi** là fitness của bộ trọng số chính sách. Bắt đầu với chính sách **tuyến tính** (10 tham số: $a = \arg\max(Ws + b)$). Fitness = return trung bình 3 lượt. Đếm **số bước tương tác với môi trường** để so sánh công bằng với policy gradient.

In [ ]:
def linear_policy(theta):
    W, b = theta[:8].reshape(2, 4), theta[8:]
    return lambda s: int(np.argmax(W @ s + b))


def rollout(policy, seed, max_steps=500):
    env = CartPole(max_steps)
    s, done, total = env.reset(seed=seed), False, 0.0
    while not done:
        s, r, done = env.step(policy(s))
        total += r
    return total


def evaluate(policy, episodes=20):
    return np.mean([rollout(policy, 10_000 + i) for i in range(episodes)])


r = np.random.default_rng(0)
t0, steps, best, best_theta = time.perf_counter(), 0, -1.0, None
for i in range(200):                                                   # tìm kiếm ngẫu nhiên
    theta = r.normal(0, 1, 10)
    returns = [rollout(linear_policy(theta), 100 * i + k) for k in range(3)]
    steps += sum(returns)
    if np.mean(returns) > best:
        best, best_theta = np.mean(returns), theta
print(f"tìm kiếm ngẫu nhiên: fitness tốt nhất = {best:.0f} | đánh giá = {evaluate(linear_policy(best_theta)):.0f} | "
      f"{int(steps):,} bước | {time.perf_counter() - t0:.0f}s")

population = r.normal(0, 1, (50, 10))
t0, steps, history = time.perf_counter(), 0, []
for generation in range(15):
    returns = np.array([[rollout(linear_policy(th), 1000 * generation + k) for k in range(3)] for th in population])
    steps += returns.sum()
    f = returns.mean(1)
    history.append((generation, f.max(), f.mean(), int(steps)))
    elite = population[np.argsort(-f)[:10]]
    if f.mean() >= 475:
        break
    children = [np.where(r.random(10) < 0.5, *elite[r.choice(10, 2, replace=False)]) + r.normal(0, 0.1, 10) for _ in range(40)]
    population = np.vstack([elite, children])
best_ga = population[0]
for g, fmax, fmean, st in history:
    print(f"GA thế hệ {g:2d}: fitness tốt nhất = {fmax:5.0f} | trung bình quần thể = {fmean:5.1f} | tổng bước = {st:,}")
print(f"GA: đánh giá cá thể tốt nhất = {evaluate(linear_policy(best_ga)):.0f} | {time.perf_counter() - t0:.0f}s")

Chính sách tuyến tính đã đủ giữ thăng bằng — và chỉ cần **vài thế hệ** để GA tìm ra (thậm chí tìm kiếm ngẫu nhiên cũng tìm được bộ trọng số khá tốt): không gian 10 chiều đủ nhỏ để "đoán" hiệu quả. Đây là lời nhắc quan trọng (Mania et al., 2018): trên bài toán đơn giản, phương pháp đơn giản có thể cạnh tranh với deep RL — luôn có baseline.

### Evolution Strategies (ES) với mạng nơ-ron

Với nhiều tham số hơn (MLP 4 → 16 → 2: 114 tham số), dùng **Evolution Strategies** (Salimans et al., 2017): ước lượng gradient của return kỳ vọng bằng nhiễu ngẫu nhiên — không cần lan truyền ngược:

$$\nabla_\theta\,\mathbb E_{\varepsilon\sim\mathcal N(0, I)}[F(\theta + \sigma\varepsilon)] \approx \frac{1}{2n\sigma}\sum_{i=1}^{n}\left[F(\theta + \sigma\varepsilon_i) - F(\theta - \sigma\varepsilon_i)\right]\varepsilon_i$$

(lấy mẫu đối xứng $\pm\varepsilon$ giảm phương sai; chuẩn hoá return theo hạng/z-score cho ổn định). Mỗi cặp nhiễu đánh giá **độc lập** → song song hoá hoàn hảo trên hàng nghìn CPU.

In [ ]:
def mlp_policy(theta):
    W1, b1 = theta[:64].reshape(16, 4), theta[64:80]
    W2, b2 = theta[80:112].reshape(2, 16), theta[112:114]
    return lambda s: int(np.argmax(W2 @ np.tanh(W1 @ s + b1) + b2))


r = np.random.default_rng(0)
theta = r.normal(0, 0.1, 114)
sigma, lr, n_pairs, steps = 0.1, 0.03, 20, 0
t0 = time.perf_counter()
for iteration in range(1, 31):
    eps = r.normal(0, 1, (n_pairs, 114))
    seeds = r.integers(1_000_000_000, size=n_pairs)                    # cặp ±ε dùng cùng seed môi trường
    R_plus = np.array([rollout(mlp_policy(theta + sigma * e), sd) for e, sd in zip(eps, seeds)])
    R_minus = np.array([rollout(mlp_policy(theta - sigma * e), sd) for e, sd in zip(eps, seeds)])
    steps += R_plus.sum() + R_minus.sum()
    A = np.r_[R_plus, R_minus]
    A = (A - A.mean()) / (A.std() + 1e-8)
    theta += lr / (2 * n_pairs * sigma) * ((A[:n_pairs] - A[n_pairs:]) @ eps)
    if iteration % 5 == 0:
        print(f"ES vòng {iteration:2d}: return TB các cá thể nhiễu = {np.r_[R_plus, R_minus].mean():5.1f} | "
              f"đánh giá θ = {evaluate(mlp_policy(theta), 10):5.0f} | tổng bước = {int(steps):,}")
print(f"{time.perf_counter() - t0:.0f}s")

ES đưa chính sách MLP lên mức tối đa khi đánh giá chỉ sau vài chục nghìn bước — ít hơn REINFORCE ở bài 04 (hàng trăm nghìn bước) trên bài toán nhỏ này, và không cần gradient. Ở quy mô lớn, ES của OpenAI đạt kết quả ngang A3C trên Atari/MuJoCo nhờ chạy song song trên hàng nghìn CPU; nhược điểm là hiệu quả mẫu giảm khi số tham số rất lớn (ước lượng gradient bằng nhiễu ngẫu nhiên trong không gian hàng triệu chiều).

## 4. Khi nào dùng tiến hoá?

| Nên dùng | Không nên dùng |
|---|---|
| Hàm mục tiêu không khả vi, rời rạc, hộp đen (mô phỏng, phần cứng, cấu hình) | Có gradient tốt (huấn luyện mạng lớn có giám sát) |
| Phần thưởng thưa/trễ, chỉ biết kết quả cuối | Mỗi lần đánh giá rất đắt và ít ngân sách → tối ưu Bayes (chương 16) |
| Song song hoá dễ (mỗi cá thể độc lập) | Bài toán có cấu trúc khai thác được (DP, quy hoạch tuyến tính) |
| Tìm **kiến trúc**/siêu tham số: NEAT, NAS, Population Based Training | Không gian tham số cực lớn với ngân sách đánh giá nhỏ |

Ứng dụng: thiết kế ăng-ten của NASA (ST5), tối ưu lịch/lộ trình, Population Based Training (DeepMind) tinh chỉnh siêu tham số trong lúc train, AlphaStar dùng "giải đấu" quần thể tác tử, tìm kiếm kiến trúc mạng (AmoebaNet).

## 🧠 Kiểm tra nhanh

1. Mô tả các bước của thuật toán di truyền. Elitism để làm gì?
2. Vì sao không có đột biến thì GA dễ hội tụ sớm? Đột biến quá lớn thì sao?
3. Neuroevolution khác policy gradient thế nào? Cần gì từ môi trường?
4. Viết ước lượng gradient của Evolution Strategies. Vì sao dùng nhiễu đối xứng?
5. Khi nào nên dùng tiến hoá thay vì gradient?

## 📌 Tóm tắt

- Tiến hoá = quần thể + chọn lọc + lai ghép + đột biến; chỉ cần đánh giá được fitness; đột biến điều khiển khám phá.
- GA tìm lời giải (gần) tối ưu cho bài toán tổ hợp tốt hơn tham lam và ngẫu nhiên cùng ngân sách, nhưng không đảm bảo tối ưu.
- Neuroevolution/ES tối ưu chính sách không cần gradient, song song hoá tốt; trên CartPole cạnh tranh với deep RL — luôn so với baseline đơn giản.

## 📚 Tài liệu tham khảo

- Holland (1975) — *Adaptation in Natural and Artificial Systems*; Goldberg (1989) — *Genetic Algorithms in Search, Optimization, and Machine Learning*
- Salimans et al. (2017) — *Evolution Strategies as a Scalable Alternative to Reinforcement Learning*
- Such et al. (2017) — *Deep Neuroevolution: Genetic Algorithms Are a Competitive Alternative for Training Deep Neural Networks for RL*
- Stanley & Miikkulainen (2002) — *Evolving Neural Networks through Augmenting Topologies* (NEAT)
- Mania, Guy & Recht (2018) — *Simple random search provides a competitive approach to reinforcement learning*
- Jaderberg et al. (2017) — *Population Based Training of Neural Networks*